In [ ]:
# Célula 0 — Imports e configuração
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from google.cloud import bigquery
from IPython.display import display, HTML

COLORS = {
    "fluxo_pa":   "#6B46C1",
    "fluxo_tri":  "#EAB308",
    "status_no_prazo":     "#3B82F6",
    "status_atencao":      "#F97316",
    "status_critico":      "#DC2626",
    "pipeline_ok":         "#6B7280",
    "pipeline_atencao":    "#F97316",
    "pipeline_critico":    "#DC2626",
    "neutro_escuro":       "#374151",
    "neutro_claro":        "#E5E7EB",
}
TEMPLATE = "plotly_white"
FALLBACK_LEAD_TIME = 120

CONFIG = {
    "janela_meses":          12,
    "threshold_atraso_critico":  16,
    "threshold_pipeline_critico":  0.60,
    "threshold_pipeline_atencao":  0.20,
}

PROJECT_ID = "insider-data-lake"
client = bigquery.Client(project=PROJECT_ID)


In [ ]:
# Célula 1 — Base de capacidade (fornecedores, produtos, lead time teórico, mark-up)
# ⚠️  REFERÊNCIA COMPLEMENTAR: Esta célula é usada apenas para enriquecer com lead_time_teorico
# Não é utilizada para cálculos de deadlines negociados (que vêm de muninn).

SQL_CAPACITY = """
WITH
full_price AS (
    SELECT *
    FROM `insider-data-lake.integrated.muninn_products`
),
fabric_costs AS(
    SELECT 
    mfs.id AS fabric_sku_id,
    mfs.fabric_id,
    mfs.knitting_factory_id,
    mfs.sku AS fabric_sku,
    mfs.invoice_fabric_name AS factory_fabric_name,
    mfs.unit_price,
    mfs.minimum_volume_per_order,
    mfs.multiple_volume_per_order,
    mf.name AS fabric_name,
    mf.article_id,
    ma.name AS article_name,
    ma.unit AS article_unit,
    mkf.supplier_id,
    ms.alias AS knitting_factory_name,
    FROM `insider-data-lake.integrated.muninn_fabric_skus` AS mfs
    LEFT JOIN `insider-data-lake.integrated.muninn_fabrics` AS mf ON mf.id = mfs.fabric_id
    LEFT JOIN `insider-data-lake.integrated.muninn_articles` AS ma ON ma.id = mf.article_id
    LEFT JOIN `insider-data-lake.integrated.muninn_knitting_factories` AS mkf ON mkf.id = mfs.knitting_factory_id
    LEFT JOIN `insider-data-lake.integrated.muninn_suppliers` AS ms ON ms.id = mkf.supplier_id
    WHERE mfs.status IN ('available')
    ),
fabric_min_max_cost AS (
    SELECT
    fc.fabric_id,
    fc.fabric_name,
    MIN(fc.unit_price) AS min_fabric_cost,
    MAX(fc.unit_price) AS max_fabric_cost,
    COUNT(DISTINCT fc.knitting_factory_id) AS number_knitting_factories,
    ARRAY_AGG(DISTINCT fc.knitting_factory_name) AS knitting_factories_names,
    FROM fabric_costs AS fc
    GROUP BY fc.fabric_id,
    fc.fabric_name
),
article_sku AS (
SELECT
    mpsf.product_sku_id,
    mps.sku,
    mps.sku_name,
    mps.product_id,
    s.sku_state,
    s.gender,
    s.color,
    s.size,
    s.product_name,
    mpsf.fabric_id,
    mf.name AS fabric_name,
    mpsf.consumption,
    fc.min_fabric_cost AS min_fabric_unitary_cost,
    fc.max_fabric_cost AS max_fabric_unitary_cost,
    fc.min_fabric_cost * mpsf.consumption AS min_fabric_cost,
    fc.max_fabric_cost * mpsf.consumption AS max_fabric_cost,
    ma.unit AS article_unit,
    ma.name AS article_name,
    mf.article_id,
    fc.number_knitting_factories,
    fc.knitting_factories_names
FROM `insider-data-lake.integrated.muninn_product_skus_fabrics` AS mpsf
LEFT JOIN `insider-data-lake.integrated.muninn_fabrics` AS mf ON mf.id = mpsf.fabric_id
LEFT JOIN `insider-data-lake.integrated.muninn_articles` AS ma ON ma.id = mf.article_id
LEFT JOIN `insider-data-lake.integrated.muninn_product_skus` AS mps ON mps.product_sku_id = mpsf.product_sku_id
LEFT JOIN `insider-data-lake.integrated.skus` AS s ON mps.sku = s.sku
LEFT JOIN fabric_min_max_cost AS fc ON fc.fabric_id = mpsf.fabric_id
),
sku_fabric_costs AS (
    SELECT 
        a_sku.sku, a_sku.sku_state, a_sku.product_id,
        SUM(a_sku.min_fabric_cost) AS min_fabric_cost,
        SUM(a_sku.max_fabric_cost) AS max_fabric_cost,
        STRING_AGG(DISTINCT article_name, ',' ORDER BY article_name) AS article_names,
    FROM article_sku AS a_sku
    GROUP BY a_sku.sku, a_sku.sku_state, a_sku.product_id
),
article_freq AS (
    SELECT product_id, article_names, COUNT(*) AS freq
    FROM sku_fabric_costs GROUP BY product_id, article_names
),
top_article AS (
    SELECT product_id,
        ARRAY_AGG(article_names ORDER BY freq DESC LIMIT 1)[OFFSET(0)] AS most_common_article_names
    FROM article_freq GROUP BY product_id
),
avg_fabric_cost_prod AS (
    SELECT fc.product_id,
        AVG(fc.max_fabric_cost) AS max_fabric_cost,
        AVG(fc.min_fabric_cost) AS min_fabric_cost,
        t.most_common_article_names AS article_names
    FROM sku_fabric_costs AS fc
    LEFT JOIN top_article AS t ON fc.product_id = t.product_id
    GROUP BY fc.product_id, t.most_common_article_names
),
costs AS (
    SELECT
        amp.product_id, p.product_name,
        amp.apparel_manufacturer_id, amp.is_finished_product,
        amp.manufacturer_cost as manufacture_cost,
        fc.min_fabric_cost, fc.max_fabric_cost, fc.article_names,
        CASE WHEN amp.is_finished_product = True THEN amp.manufacturer_cost
             ELSE amp.manufacturer_cost + fc.max_fabric_cost END AS manufacturing_cost,
    FROM `insider-data-lake.integrated.muninn_apparel_manufacturers_products` AS amp
    LEFT JOIN avg_fabric_cost_prod AS fc ON fc.product_id = amp.product_id
    LEFT JOIN `insider-data-lake.integrated.muninn_products` AS p ON p.product_id = amp.product_id
    WHERE amp.status IN ('available','approved','incubation')
),
base_intermediaria AS (
    SELECT
        ampup.apparel_manufacturer_production_unit_id,
        am.supplier_id, amp.apparel_manufacturer_id,
        s.alias, s.city, s.state, s.created_at AS date_supplier_creation,
        p.product_id, p.product_name,
        amp.is_finished_product, amp.order_minimum_volume, amp.lead_time,
        ampu.apparel_manufacturer_cell_number,
        MAX(ampup.weekly_maximum_productive_capacity) OVER (
            PARTITION BY ampup.apparel_manufacturer_production_unit_id
        ) AS max_capacity,
        ampup.weekly_maximum_productive_capacity,
        fp.full_price, amp.status AS status_cell,
        4*ampup.weekly_maximum_productive_capacity AS monthly_capacity,
        4*MAX(ampup.weekly_maximum_productive_capacity) OVER (
            PARTITION BY ampup.apparel_manufacturer_production_unit_id
        ) AS cell_max_monthly_capacity,
        COUNT(DISTINCT am.supplier_id) OVER (PARTITION BY p.product_id) AS num_suppliers_per_product,
    FROM `insider-data-lake.integrated.muninn_apparel_manufacturer_production_units_products` AS ampup
    LEFT JOIN `insider-data-lake.integrated.muninn_apparel_manufacturers_products` AS amp
        ON amp.id = ampup.apparel_manufacturer_product_id
    LEFT JOIN `insider-data-lake.integrated.muninn_products` AS p ON p.product_id = amp.product_id
    LEFT JOIN `insider-data-lake.integrated.muninn_apparel_manufacturers` AS am ON am.id = amp.apparel_manufacturer_id
    LEFT JOIN `insider-data-lake.integrated.muninn_apparel_manufacturer_production_units` AS ampu
        ON ampu.id = ampup.apparel_manufacturer_production_unit_id
    LEFT JOIN `insider-data-lake.integrated.muninn_suppliers` AS s ON s.id = am.supplier_id
    LEFT JOIN full_price AS fp ON p.product_name = fp.product_name
    WHERE amp.status IN ('available','approved','incubation')
        AND ampup.weekly_maximum_productive_capacity > 0
),
cell_products AS (
    SELECT apparel_manufacturer_production_unit_id,
        COUNT(DISTINCT b.product_name) AS n_products_in_cell,
        STRING_AGG(DISTINCT b.product_name, ', ') AS products_in_cell
    FROM base_intermediaria AS b
    GROUP BY apparel_manufacturer_production_unit_id
),
sku_data AS (
    SELECT ps.sku, ps.product_sku_id, ps.sku_name, sku_d.sku_state, sku_d.product_name,
        sku_d.family, sku_d.category, psf.fabric_id, a.name AS article_name
    FROM `insider-data-lake.integrated.muninn_product_skus` AS ps
    LEFT JOIN `insider-data-lake.integrated.muninn_product_skus_fabrics` AS psf ON ps.product_sku_id = psf.product_sku_id
    LEFT JOIN `insider-data-lake.integrated.muninn_fabrics` AS f ON psf.fabric_id = f.id
    LEFT JOIN `insider-data-lake.integrated.muninn_articles` AS a ON f.article_id = a.id
    LEFT JOIN `insider-data-lake.integrated.skus` AS sku_d ON sku_d.sku = ps.sku
    ORDER BY ps.product_sku_id, psf.fabric_id
),
dpi AS (
    SELECT product_name, SUM(treated_generated_revenue) AS treated_generated_revenue
    FROM `insider-data-lake.sop_silver.demand_prediction_input`
    WHERE DATE(reference_date) >= DATE_SUB(DATE_TRUNC(CURRENT_DATE(), MONTH), INTERVAL 3 MONTH)
        AND DATE(reference_date) <  DATE_TRUNC(CURRENT_DATE(), MONTH)
        AND product_name IS NOT NULL
    GROUP BY product_name
),
revenue_totals AS (
    SELECT product_name, treated_generated_revenue,
        SUM(treated_generated_revenue) OVER () AS total_treated_generated_revenue
    FROM dpi
),
cum AS (
    SELECT product_name, treated_generated_revenue, total_treated_generated_revenue,
        SUM(treated_generated_revenue) OVER (
            ORDER BY treated_generated_revenue DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cum_treated_generated_revenue
    FROM revenue_totals
),
abc_curve AS (
    SELECT product_name, treated_generated_revenue,
        SAFE_DIVIDE(cum_treated_generated_revenue, total_treated_generated_revenue) AS cum_share,
        CASE
            WHEN SAFE_DIVIDE(cum_treated_generated_revenue, total_treated_generated_revenue) <= 0.8 THEN 'A'
            WHEN SAFE_DIVIDE(cum_treated_generated_revenue, total_treated_generated_revenue) <= 0.95 THEN 'B'
            ELSE 'C'
        END AS tag_abc
    FROM cum
),
freqs AS (
    SELECT product_name, sku_state, article_name, family, category, COUNT(*) AS freq
    FROM sku_data GROUP BY product_name, sku_state, article_name, family, category
),
product_data AS (
    SELECT f.product_name, p.product_id,
        CASE WHEN SUM(CASE WHEN f.sku_state = 'ativo_perene' THEN 1 ELSE 0 END) > 0
             THEN 'ativo_perene'
             ELSE ARRAY_AGG(f.sku_state ORDER BY freq DESC LIMIT 1)[OFFSET(0)] END AS product_state,
        ARRAY_TO_STRING(ARRAY_AGG(DISTINCT f.article_name), ', ') AS article_name,
        ARRAY_AGG(f.family ORDER BY freq DESC LIMIT 1)[OFFSET(0)] AS family,
        ARRAY_AGG(f.category ORDER BY freq DESC LIMIT 1)[OFFSET(0)] AS category,
        abc.tag_abc
    FROM freqs AS f
    LEFT JOIN `insider-data-lake.integrated.muninn_products` AS p ON f.product_name = p.product_name
    LEFT JOIN abc_curve AS abc ON abc.product_name = f.product_name
    GROUP BY f.product_name, p.product_id, abc.tag_abc
)
SELECT
    b.*,
    c.manufacturing_cost, c.article_names,
    SAFE_DIVIDE(b.full_price, c.manufacturing_cost) AS mark_up,
    MIN(c.manufacturing_cost) OVER (PARTITION BY b.product_id, b.is_finished_product) AS min_manufacturing_cost,
    cp.n_products_in_cell, cp.products_in_cell,
    pd.tag_abc, pd.product_state,
FROM base_intermediaria AS b
LEFT JOIN cell_products AS cp ON b.apparel_manufacturer_production_unit_id = cp.apparel_manufacturer_production_unit_id
LEFT JOIN costs AS c ON b.apparel_manufacturer_id = c.apparel_manufacturer_id
    AND b.product_id = c.product_id AND b.is_finished_product = c.is_finished_product
LEFT JOIN product_data AS pd ON b.product_id = pd.product_id
WHERE pd.product_state NOT IN ('desativado')
"""

print("⏳ Carregando base de capacidade...")
df_capacity = client.query(SQL_CAPACITY).to_dataframe()
print(f"✅ df_capacity: {len(df_capacity):,} linhas | {df_capacity.shape[1]} colunas")
df_capacity.head(3)

In [ ]:
# Célula 2 – Base principal de OPs (fonte: supply_production_orders + muninn pivot)\n\nSQL_OPS = """\nWITH muninn_pivot AS (\n  SELECT\n    order_code,\n    MIN(CASE WHEN status = 'order_request_validation' THEN DATE(ingestion_date) END) AS dt_muninn_order_request_validation,\n    MIN(CASE WHEN status = 'finished' THEN DATE(ingestion_date) END) AS dt_muninn_finished\n  FROM `insider-lake-sensitive.landing_br.muninn_production_orders_raw`\n  WHERE order_code LIKE 'OPF%' AND ingestion_date >= '2025-01-01'\n  GROUP BY 1\n),\nbase_ops AS (\n  SELECT\n    spo.production_order_code,\n    spo.production_order_type,\n    spo.cycle_name,\n    spo.apparel_manufacturer_alias AS supplier_name,\n    spo.product_name,\n    spo.is_finished_product_order,\n    spo.planned_quantity_op,\n    DATE(spo.planned_production_delivery_date) AS data_entrega_planejada,\n    spo.stamp_created_production_order,\n    spo.dt_largest_entry_warehouse,\n    spo.stamp_stage_order_request_validation,\n    spo.stamp_stage_waiting_fabric_arrival,\n    spo.stamp_stage_fabric_validation_and_pre_cutting,\n    spo.stamp_stage_cut_fabric_and_sewing_process,\n    spo.stamp_stage_quality_inspection,\n    spo.stamp_stage_items_delivery_and_invoicing,\n    spo.stamp_stage_finished,\n    DATE_DIFF(m.dt_muninn_finished, m.dt_muninn_order_request_validation, DAY) AS lead_time_realizado_dias,\n    DATE_DIFF(m.dt_muninn_finished, spo.planned_production_delivery_date, DAY) AS atraso_realizado_dias,\n    CASE WHEN spo.is_finished_product_order THEN 'Produto acabado' ELSE 'Triangulação' END AS fluxo\n  FROM `insider-lake-sensitive.integrated_br.supply_production_orders` spo\n  LEFT JOIN muninn_pivot m ON spo.production_order_code = m.order_code\n  WHERE spo.planned_production_delivery_date >= '2025-01-01'\n)\nSELECT * FROM base_ops\n"""\n\nprint("⏳ Carregando base principal de OPs...")\ndf_ops_raw = client.query(SQL_OPS).to_dataframe()\nprint(f"✅ df_ops_raw: {len(df_ops_raw):,} linhas | {df_ops_raw.shape[1]} colunas")

In [ ]:
# Célula 3 — Pré-processamento: filtros, lead time realizado, etapas calculadas

# --- 3.1 Filtros conforme metodologia do relatório original ---
df_ops = df_ops_raw.copy()

# Apenas production_order_type = 'committed'
df_ops = df_ops[df_ops["production_order_type"] == "committed"]

# Remover ciclos B2B e EPA
df_ops = df_ops[~df_ops["cycle_name"].str.contains("B2B|EPA", na=False, case=False)]

# Lead time em dias já calculado do muninn
df_ops["lead_time_realizado"] = df_ops["lead_time_realizado_dias"].fillna(0).astype(int)

# Remover lead times nulos ou negativos
df_ops = df_ops[(df_ops["lead_time_realizado"].notna()) & (df_ops["lead_time_realizado"] > 0)]

print(f"OPs após filtros: {len(df_ops):,}")
print(f"  Triangulação: {(~df_ops['is_finished_product_order']).sum():,}")
print(f"  Produto acabado: {df_ops['is_finished_product_order'].sum():,}")

# --- 3.2 Etapas de tempo calculadas a partir dos stamps ---
stamp_cols = [
    "stamp_stage_order_request_validation",
    "stamp_stage_waiting_fabric_arrival",
    "stamp_stage_fabric_validation_and_pre_cutting",
    "stamp_stage_cut_fabric_and_sewing_process",
    "stamp_stage_quality_inspection",
    "stamp_stage_items_delivery_and_invoicing",
    "stamp_stage_finished",
]
for col in stamp_cols:
    df_ops[col] = pd.to_datetime(df_ops[col], utc=True, errors='coerce')

def days_between(df, col_start, col_end):
    return (df[col_end] - df[col_start]).dt.days.clip(lower=0)

df_ops["etapa_pre_costura"] = days_between(
    df_ops,
    "stamp_stage_order_request_validation",
    "stamp_stage_cut_fabric_and_sewing_process",
)
df_ops["etapa_costura_inspecao"] = days_between(
    df_ops,
    "stamp_stage_cut_fabric_and_sewing_process",
    "stamp_stage_items_delivery_and_invoicing",
)
df_ops["etapa_pos_inspecao"] = days_between(
    df_ops,
    "stamp_stage_items_delivery_and_invoicing",
    "stamp_stage_finished",
)

# --- 3.3 Enriquecer com lead time teórico da base de capacidade ---
capacity_lt = (
    df_capacity[["alias", "product_name", "is_finished_product", "lead_time", "tag_abc"]]
    .drop_duplicates()
    .rename(columns={
        "alias": "supplier_name",
        "lead_time": "lead_time_teorico",
        "is_finished_product": "is_finished_product_order",
    })
)
capacity_lt["is_finished_product_order"] = capacity_lt["is_finished_product_order"].astype(bool)

df_ops = df_ops.merge(
    capacity_lt,
    how="left",
    on=["supplier_name", "product_name", "is_finished_product_order"],
)

df_ops["desvio_lt"] = df_ops["lead_time_realizado"] - df_ops["lead_time_teorico"]
df_ops["dentro_do_prazo"] = df_ops["lead_time_realizado"] <= FALLBACK_LEAD_TIME

# --- 3.4 Buckets de volume ---
bins = [0, 200, 500, 1000, 2000, 5000, float("inf")]
labels = ["<200", "200–499", "500–999", "1000–1999", "2000–4999", "5000+"]
df_ops["volume_bucket"] = pd.cut(df_ops["planned_quantity_op"], bins=bins, labels=labels, right=False)

# Separar os dois fluxos para facilitar uso nas células seguintes
df_tri = df_ops[~df_ops["is_finished_product_order"]].copy()
df_pa  = df_ops[df_ops["is_finished_product_order"]].copy()

print(f"\n✅ Pré-processamento completo.")
print(f"   df_tri (triangulação): {len(df_tri):,} OPs")
print(f"   df_pa  (produto acabado): {len(df_pa):,} OPs")
print(f"   Cobertura lead time teórico: {df_ops['lead_time_teorico'].notna().mean():.1%}")


In [ ]:
# Célula 3.1 – Diagnóstico de Qualidade da Base
from IPython.display import display, HTML

n_total = len(df_ops)
qualidade = {
    "Total de OPs": n_total,
    "% com data_entrega_planejada": f"{(df_ops['data_entrega_planejada'].notna().sum() / n_total * 100):.1f}%",
}

display(HTML(f"""
<div style='font-family:-apple-system,sans-serif'>
  <h3 style='color:#374151;margin-bottom:8px'>📋 Diagnóstico de Qualidade</h3>
  <p>Total de OPs: {n_total:,}</p>
</div>
"""))

In [ ]:
# Célula 4 — Nível 1: KPI Cards
from IPython.display import display, HTML

lt_mediana_geral = df_ops["lead_time_realizado"].median()
lt_mediana_tri   = df_tri["lead_time_realizado"].median()
lt_mediana_pa    = df_pa["lead_time_realizado"].median()
pct_dentro_prazo = df_ops["dentro_do_prazo"].mean() * 100

def kpi_card(label, value, unit="", color="#1B2A4A", sub=None):
    sub_html = f"<div style='font-size:13px;color:#888;margin-top:4px'>{sub}</div>" if sub else ""
    return f'''
    <div style='display:inline-block;background:#f9f9f9;border:1px solid #ddd;
                border-radius:8px;padding:20px 28px;margin:8px;min-width:160px;text-align:center'>
        <div style='font-size:13px;color:#666;font-weight:600'>{label}</div>
        <div style='font-size:36px;font-weight:700;color:{color}'>{value}<span style='font-size:16px'>{unit}</span></div>
        {sub_html}
    </div>'''

alert_color = COLORS["status_critico"] if lt_mediana_pa > FALLBACK_LEAD_TIME else COLORS["status_no_prazo"]

cards_html = "".join([
    kpi_card("Mediana Geral", f"{lt_mediana_geral:.0f}", "d", COLORS["status_no_prazo"],
             f"Target: {FALLBACK_LEAD_TIME}d"),
    kpi_card("Mediana Triangulação", f"{lt_mediana_tri:.0f}", "d", COLORS["pipeline_ok"]),
    kpi_card("Mediana Produto Acabado", f"{lt_mediana_pa:.0f}", "d", alert_color),
    kpi_card("Dentro do Prazo (120d)", f"{pct_dentro_prazo:.1f}", "%",
             COLORS["status_no_prazo"] if pct_dentro_prazo >= 65 else COLORS["status_atencao"]),
])
display(HTML(f"<div style='font-family:sans-serif'><h3 style='color:#1B2A4A'>📊 Nível 1 — Visão Executiva</h3>{cards_html}</div>"))


In [ ]:
# Célula 5 — Comparativo Triangulação vs Produto Acabado (Mediana e P75)
summary = pd.DataFrame({
    "Fluxo": ["Triangulação", "Produto Acabado"],
    "Mediana": [
        df_tri["lead_time_realizado"].median(),
        df_pa["lead_time_realizado"].median(),
    ],
    "P75": [
        df_tri["lead_time_realizado"].quantile(0.75),
        df_pa["lead_time_realizado"].quantile(0.75),
    ],
    "% Dentro 120d": [
        df_tri["dentro_do_prazo"].mean() * 100,
        df_pa["dentro_do_prazo"].mean() * 100,
    ],
})

fig = go.Figure()
fig.add_bar(
    name="Mediana", x=summary["Fluxo"], y=summary["Mediana"],
    marker_color=[COLORS["status_no_prazo"], COLORS["status_no_prazo"]],
    text=summary["Mediana"].round(0).astype(int), textposition="outside",
)
fig.add_bar(
    name="P75", x=summary["Fluxo"], y=summary["P75"],
    marker_color=[COLORS["status_atencao"], COLORS["status_critico"]],
    text=summary["P75"].round(0).astype(int), textposition="outside",
)
fig.add_hline(
    y=FALLBACK_LEAD_TIME, line_dash="dash", line_color="gray",
    annotation_text=f"Target {FALLBACK_LEAD_TIME}d", annotation_position="right",
)
fig.update_layout(
    title="Lead Time por Fluxo — Mediana e P75",
    barmode="group", template=TEMPLATE,
    yaxis_title="Dias", xaxis_title="",
    legend_title="Métrica",
)
fig.show()


In [ ]:
# Célula 6 — Decomposição de tempo por etapa (apenas triangulação, onde stamps existem)
etapas_summary = df_tri.groupby("supplier_name").agg(
    pre_costura=("etapa_pre_costura", "median"),
    costura_inspecao=("etapa_costura_inspecao", "median"),
    pos_inspecao=("etapa_pos_inspecao", "median"),
    n_ops=("production_order_code", "count"),
).reset_index()

# Focar nos top 15 fornecedores por volume
etapas_summary = etapas_summary.nlargest(15, "n_ops")
etapas_summary = etapas_summary.sort_values("pre_costura", ascending=False)

fig = go.Figure()
fig.add_bar(name="Pré-costura", x=etapas_summary["supplier_name"], y=etapas_summary["pre_costura"],
            marker_color=COLORS["status_critico"])
fig.add_bar(name="Costura + Inspeção", x=etapas_summary["supplier_name"], y=etapas_summary["costura_inspecao"],
            marker_color=COLORS["status_atencao"])
fig.add_bar(name="Pós-inspeção", x=etapas_summary["supplier_name"], y=etapas_summary["pos_inspecao"],
            marker_color=COLORS["neutro_claro"])
fig.update_layout(
    title="Decomposição de Lead Time por Etapa — Top 15 Fornecedores (Triangulação)",
    barmode="stack", template=TEMPLATE,
    yaxis_title="Dias (Mediana)", xaxis_title="Fornecedor",
    xaxis_tickangle=-30,
)
fig.show()


In [ ]:
# Célula 7 — Nível 2: Aderência ao Lead Time Teórico por Fornecedor
aderencia = (
    df_ops[df_ops["lead_time_teorico"].notna()]
    .groupby(["supplier_name", "is_finished_product_order"])
    .agg(
        desvio_mediano=("desvio_lt", "median"),
        n_ops=("production_order_code", "count"),
        lt_realizado=("lead_time_realizado", "median"),
        lt_teorico=("lead_time_teorico", "median"),
    )
    .reset_index()
)
aderencia["fluxo"] = aderencia["is_finished_product_order"].map(
    {True: "Produto Acabado", False: "Triangulação"}
)
# Filtrar min 5 OPs e ordenar por desvio
aderencia = aderencia[aderencia["n_ops"] >= 5].sort_values("desvio_mediano", ascending=False)

def desvio_color(d):
    if d > 30: return COLORS["status_critico"]
    if d > 10: return COLORS["status_atencao"]
    return COLORS["status_no_prazo"]

colors = aderencia["desvio_mediano"].apply(desvio_color).tolist()

fig = go.Figure(go.Bar(
    x=aderencia["supplier_name"] + " (" + aderencia["fluxo"].str[0] + ")",
    y=aderencia["desvio_mediano"],
    marker_color=colors,
    text=aderencia["desvio_mediano"].round(0).astype(int),
    textposition="outside",
    customdata=aderencia[["lt_realizado", "lt_teorico", "n_ops"]].values,
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Desvio: %{y:.0f}d<br>"
        "Realizado: %{customdata[0]:.0f}d | Teórico: %{customdata[1]:.0f}d<br>"
        "n OPs: %{customdata[2]}<extra></extra>"
    ),
))
fig.add_hline(y=30, line_dash="dash", line_color=COLORS["status_critico"],
              annotation_text="Threshold 30d", annotation_position="right")
fig.add_hline(y=0, line_color="black", line_width=0.5)
fig.update_layout(
    title="Desvio Mediano: Lead Time Realizado vs. Teórico por Fornecedor",
    template=TEMPLATE, yaxis_title="Desvio (dias)", xaxis_tickangle=-35,
)
fig.show()


In [ ]:
# Célula 8 — Matriz: Lead Time Mediano por Bucket de Volume e Fluxo
matrix = (
    df_ops.groupby(["volume_bucket", "is_finished_product_order"])["lead_time_realizado"]
    .agg(["median", "count"])
    .reset_index()
)
matrix.columns = ["volume_bucket", "is_finished_product_order", "mediana", "n_ops"]
matrix["fluxo"] = matrix["is_finished_product_order"].map(
    {True: "Produto Acabado", False: "Triangulação"}
)
matrix = matrix[matrix["n_ops"] >= 3]

fig = px.bar(
    matrix, x="volume_bucket", y="mediana", color="fluxo", barmode="group",
    text="mediana",
    color_discrete_map={"Triangulação": COLORS["status_no_prazo"], "Produto Acabado": COLORS["status_atencao"]},
    labels={"mediana": "Lead Time Mediano (dias)", "volume_bucket": "Faixa de Volume"},
    title="Lead Time Mediano por Faixa de Volume e Fluxo",
    template=TEMPLATE,
)
fig.add_hline(y=FALLBACK_LEAD_TIME, line_dash="dash", line_color="gray",
              annotation_text=f"Target {FALLBACK_LEAD_TIME}d")
fig.update_traces(texttemplate="%{text:.0f}d", textposition="outside")
fig.show()


In [ ]:
# Célula 9 — Nível 3: Risco Single-Source (produtos com único fornecedor ativo)
single_source = (
    df_capacity[df_capacity["num_suppliers_per_product"] == 1]
    [[
        "product_name", "alias", "is_finished_product", "lead_time",
        "tag_abc", "product_state",
    ]]
    .drop_duplicates()
    .rename(columns={"alias": "fornecedor", "lead_time": "lt_teorico"})
)

# Enriquecer com lead time realizado mediano da base de OPs
lt_por_fornecedor_produto = (
    df_ops.groupby(["product_name", "supplier_name", "is_finished_product_order"])
    .agg(
        lt_realizado=("lead_time_realizado", "median"),
        desvio_mediano=("desvio_lt", "median"),
        n_ops=("production_order_code", "count"),
    )
    .reset_index()
    .rename(columns={"supplier_name": "fornecedor",
                     "is_finished_product_order": "is_finished_product"})
)

single_source = single_source.merge(lt_por_fornecedor_produto, how="left",
                                     on=["product_name", "fornecedor", "is_finished_product"])

def nivel_atencao(row):
    if row.get("tag_abc") in ["A", "B"]:
        if pd.notna(row.get("desvio_mediano")) and row["desvio_mediano"] > 30:
            return "🔴 Alto"
        return "🟡 Moderado"
    return "🟢 Baixo"

single_source["nivel_atencao"] = single_source.apply(nivel_atencao, axis=1)
single_source["fluxo"] = single_source["is_finished_product"].map(
    {True: "PA", False: "Tri"}
)

atencao_order = {"🔴 Alto": 0, "🟡 Moderado": 1, "🟢 Baixo": 2}
single_source["_sort"] = single_source["nivel_atencao"].map(atencao_order)
single_source = single_source.sort_values(["_sort", "tag_abc"]).drop(columns="_sort")

display_cols = [
    "product_name", "fornecedor", "fluxo", "tag_abc",
    "lt_teorico", "lt_realizado", "desvio_mediano", "n_ops", "nivel_atencao",
]
print(f"Produtos single-source: {len(single_source)}")
single_source[display_cols].style.map(
    lambda v: "background-color: #fde8e8" if v == "🔴 Alto" else
              ("background-color: #fff8e1" if v == "🟡 Moderado" else ""),
    subset=["nivel_atencao"]
)


In [ ]:
# Célula 10 — Comparação de spreads: mesmo produto + faixa de volume
spread = (
    df_ops.groupby(["product_name", "volume_bucket", "is_finished_product_order", "supplier_name"])
    .agg(lt_mediano=("lead_time_realizado", "median"), n_ops=("production_order_code", "count"))
    .reset_index()
)

# Mínimo 3 OPs por combinação, e ao menos 2 fornecedores no mesmo produto+faixa+fluxo
spread = spread[spread["n_ops"] >= 3]
contagem = (
    spread.groupby(["product_name", "volume_bucket", "is_finished_product_order"])["supplier_name"]
    .nunique()
    .reset_index(name="n_fornecedores")
)
spread = spread.merge(contagem, on=["product_name", "volume_bucket", "is_finished_product_order"])
spread = spread[spread["n_fornecedores"] >= 2]

# Calcular spread (máx - mín) por grupo usando merge explícito de min/max
spread_min = (
    spread.sort_values("lt_mediano")
    .groupby(["product_name", "volume_bucket", "is_finished_product_order"])
    .first()
    .reset_index()
    [["product_name", "volume_bucket", "is_finished_product_order", "lt_mediano", "supplier_name"]]
    .rename(columns={"lt_mediano": "lt_min", "supplier_name": "fornecedor_rapido"})
)
spread_max = (
    spread.sort_values("lt_mediano", ascending=False)
    .groupby(["product_name", "volume_bucket", "is_finished_product_order"])
    .first()
    .reset_index()
    [["product_name", "volume_bucket", "is_finished_product_order", "lt_mediano", "supplier_name"]]
    .rename(columns={"lt_mediano": "lt_max", "supplier_name": "fornecedor_lento"})
)
spread_agg = spread_min.merge(spread_max, on=["product_name", "volume_bucket", "is_finished_product_order"])
spread_agg["spread_dias"] = spread_agg["lt_max"] - spread_agg["lt_min"]
spread_agg["fluxo"] = spread_agg["is_finished_product_order"].map(
    {True: "Produto Acabado", False: "Triangulação"}
)
spread_agg = spread_agg.sort_values("spread_dias", ascending=False).head(20)

fig = px.bar(
    spread_agg,
    x="spread_dias",
    y=spread_agg["product_name"] + " | " + spread_agg["volume_bucket"].astype(str) + " | " + spread_agg["fluxo"],
    orientation="h",
    color="spread_dias",
    color_continuous_scale=["#2ECC71", "#F28C28", "#D7263D"],
    labels={"x": "Spread de Lead Time (dias)", "y": ""},
    title="Top 20 Spreads: Diferença entre Fornecedor Mais Rápido e Mais Lento<br>(mesmo produto + faixa de volume)",
    template=TEMPLATE,
    text="spread_dias",
)
fig.update_traces(texttemplate="%{text:.0f}d")
fig.update_coloraxes(showscale=False)
fig.update_layout(height=600, yaxis={"categoryorder": "total ascending"})
fig.show()
